In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error



In [2]:
# list available files (optional sanity check)
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/train.csv
/kaggle/input/train.csv.zip
/kaggle/input/sample_submission.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/test.csv.zip
/kaggle/input/description.md
/kaggle/input/ventilator-pressure-prediction/train.csv
/kaggle/input/ventilator-pressure-prediction/train.csv.zip
/kaggle/input/ventilator-pressure-prediction/sample_submission.csv
/kaggle/input/ventilator-pressure-prediction/sample_submission.csv.zip
/kaggle/input/ventilator-pressure-prediction/test.csv
/kaggle/input/ventilator-pressure-prediction/test.csv.zip
/kaggle/input/ventilator-pressure-prediction/description.md


In [3]:
train = pd.read_csv("/kaggle/input/ventilator-pressure-prediction/train.csv")
test = pd.read_csv("/kaggle/input/ventilator-pressure-prediction/test.csv")
submission = pd.read_csv(
    "/kaggle/input/ventilator-pressure-prediction/sample_submission.csv"
)



In [4]:
# basic sanity checks (can be omitted)
print("train shape:", train.shape)
print("test shape :", test.shape)



train shape: (5432400, 8)
test shape : (603600, 7)


In [5]:
# target variable statistics
target = train["pressure"]
print(
    "Pressure stats - min:",
    target.min(),
    "max:",
    target.max(),
    "mean:",
    target.mean(),
    "std:",
    target.std(),
)



Pressure stats - min: -1.895744295 max: 64.82099174 mean: 11.218069409525942 std: 8.106473982205056


In [6]:
# combine train (without target) and test for uniform preprocessing
train_features = train.drop(["pressure"], axis=1)
combi = pd.concat([train_features, test], axis=0, ignore_index=True)



In [7]:
# remove the global 'id' column – it is not useful for modeling
combi.drop(["id"], axis=1, inplace=True)



In [8]:
# check for missing values
print("Missing values per column:\n", combi.isnull().sum())



Missing values per column:
 breath_id    0
R            0
C            0
time_step    0
u_in         0
u_out        0
dtype: int64


In [9]:
# standardize features (zero mean, unit variance)
combi = (combi - combi.mean()) / combi.std()



In [10]:
# split back into train and test portions
X = combi.iloc[: len(train), :].values.astype(np.float32)
X_test = combi.iloc[len(train) :, :].values.astype(np.float32)
y = target.values.astype(np.float32)

print("X shape:", X.shape, "X_test shape:", X_test.shape, "y shape:", y.shape)



X shape: (5432400, 6) X_test shape: (603600, 6) y shape: (5432400,)


In [11]:
# convert to torch tensors
X_tensor = torch.from_numpy(X)
X_test_tensor = torch.from_numpy(X_test)
y_tensor = torch.from_numpy(y.reshape(-1, 1))

print(
    "Tensor shapes - X:",
    X_tensor.shape,
    "y:",
    y_tensor.shape,
    "X_test:",
    X_test_tensor.shape,
)



Tensor shapes - X: torch.Size([5432400, 6]) y: torch.Size([5432400, 1]) X_test: torch.Size([603600, 6])


In [12]:
# simple linear regression model
input_size = X.shape[1]  # should be 6
output_size = 1
model = nn.Linear(input_size, output_size)

learning_rate = 0.0001
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)



In [13]:
# training loop
num_epochs = 500
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_tensor)
    loss = criterion(y_pred, y_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch+1}/{num_epochs}, loss {loss.item():.6f}")



epoch 100/500, loss 183.156921
epoch 200/500, loss 176.629456
epoch 300/500, loss 170.437317
epoch 400/500, loss 164.560898
epoch 500/500, loss 158.981918


In [14]:
# evaluate on training data (RMSE for reference)
model.eval()
with torch.no_grad():
    train_pred = model(X_tensor).cpu().numpy()
rmse = mean_squared_error(y, train_pred, squared=False)
print("Training RMSE:", rmse)



Training RMSE: 12.606648


In [15]:
# generate predictions for the test set
model.eval()
with torch.no_grad():
    test_prediction = model(X_test_tensor).cpu().numpy().ravel()
# pressure cannot be negative
test_prediction = np.clip(test_prediction, a_min=0, a_max=None)



In [16]:
# prepare submission file
submission["pressure"] = test_prediction
submission.to_csv("submission.csv", index=False)

# quick verification
print("Saved submission head:")
print(submission.head())

Saved submission head:
   id  pressure
0   1  1.895182
1   2  1.925160
2   3  1.951239
3   4  2.027584
4   5  2.105578
